In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
class LSTConvNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.conv1d = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.conv1d(X))
        out = out[:, :, :-(self.kernel_size - 1)].contiguous()
        out = self.dropout(out)
        return out
    

class LSTGruNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(X)
        return self.dropout(hidden[-1])  # [batch_size, hidden_size]


class LSTSkipGruNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_step_sizes: list[int],
        skip_out_channels: list[int],
        dropout: float = 0.2,
    ):
        super().__init__()
        assert len(skip_step_sizes) == len(skip_out_channels)
        assert len(skip_step_sizes) > 0
        self.rnn_skip_step_sizes = skip_step_sizes
        self.rnn_skip_out_channels = skip_out_channels
        self.rnn_skip_nets = nn.ModuleList()
        for i in range(len(self.rnn_skip_step_sizes)):
            skip_net = nn.GRU(
                input_size=in_channels,
                hidden_size=skip_out_channels[i],
                batch_first=True
            )
            self.rnn_skip_nets.append(skip_net)
        self.skip_dropout = nn.Dropout(dropout)

    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        skip_outputs: list[torch.Tensor] = []
        
        batch_size, _, n_timesteps = X.size()  # assume X is [batch_size, in_features, timesteps]
        for i, skip_step_size in enumerate(self.rnn_skip_step_sizes):
            # Calculate integer number of sub-sequences of length skip_step_size
            n_skip_sequences = n_timesteps // skip_step_size

            # Only keep n_skip_sequences * skip_step_size worth of skips
            # so that we can reshape later
            # [batch_size, in_features, n_skip_sequences * skip_step_size]
            skip_in = X[:, :, -n_skip_sequences*skip_step_size:].contiguous()

            # Expand last time axis into into (n_skip_sequences, skip_step_size). 
            # So last time axis is now a matrix of n_skip_sequences (rows) 
            # each of length skip_step_size (columns)
            skip_in = skip_in.view(skip_in.size(0), skip_in.size(1), n_skip_sequences, skip_step_size)

            # Permute to [batch_size, skip_step_size, n_skip_sequences, in_features]
            skip_in = skip_in.permute(0, 3, 2, 1).contiguous()

            # Collapse first batch_size and skip_step_size dimensions into single dimension
            # [bath_size * skip_step_size, n_skip_sequences, in_features]
            skip_in = skip_in.view(skip_in.size(0) * skip_in.size(1), skip_in.size(2), skip_in.size(3))

            # Apply GRU for this skip step size and keep the outputs of the final 
            # hidden state. These are the final embeddings for all phase-aligned sub-sequences, 
            # where each sub-sequence contains all timesteps that share the same phase modulo 
            # e.g. the same hour-of-day.
            # Shape is [batch_size * skip_step, rnn_skip_out_channels]
            _, skip_hidden = self.rnn_skip_nets[i](skip_in)
            skip_out = skip_hidden[-1]
    
            # Bring back the batch dimension
            skip_out = skip_out.view(batch_size, skip_step_size * skip_out.size(1))
            skip_out = self.skip_dropout(skip_out)
            skip_outputs.append(skip_out)
        
        return torch.cat(skip_outputs, dim=1)


class LSTNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        cnn_out_channels: int = 32,
        cnn_kernel_size: int = 2,
        rnn_out_channels: int = 64,
        rnn_skip_step_sizes: list[int] | None = None,
        rnn_skip_out_channels: list[int] | None = None,
        ar_window_size: int = 0,
        dropout: float = 0.2
    ):
        super().__init__()

        self.conv_net = LSTConvNet(
            in_channels=in_channels,
            out_channels=cnn_out_channels,
            kernel_size=cnn_kernel_size,
            dropout=dropout,
        )
        
        # Recurrent GRU Net
        self.rnn_net = LSTGruNet(
            input_size=cnn_out_channels,
            hidden_size=rnn_out_channels,
            dropout=dropout,
        )

        # Recurrent GRU Skip Nets
        assert len(rnn_skip_step_sizes) == len(rnn_skip_out_channels)
        self.skip_rnn_net = LSTSkipGruNet(
            in_channels=cnn_out_channels,
            skip_step_sizes=rnn_skip_step_sizes,
            skip_out_channels=rnn_skip_out_channels,
            dropout=dropout
        )
        
        # Linear layer to decode rnn + skip rnn outputs
        decoder_in_channels = rnn_out_channels + np.dot(rnn_skip_step_sizes, rnn_skip_out_channels)
        self.decoder_net = nn.Linear(in_features=decoder_in_channels, out_features=out_channels)
        
        # Highway Net
        self.target_index = -1
        self.ar_window_size = ar_window_size
        if self.ar_window_size > 0:
            self.ar_net = nn.Linear(self.ar_window_size, out_features=out_channels)
        


    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Conv net
        conv_in = X.permute(0, 2, 1).contiguous()  # [batch_size, in_channels, time_steps] 
        conv_out = self.conv_net(conv_in)          # [batch_size, cnn_out_channels, time_steps]

        # Recurrent GRU Net
        gru_in = conv_out.permute(0, 2, 1)       # [batch_size, time_steps, cnn_out_channels]
        gru_out = self.rnn_net(gru_in)           # [batch_size, rnn_out_channels]

        # Recurrent skip GRUs
        skip_out = self.skip_rnn_net(conv_out)           # [batch_size, dot(rnn_skip_step_sizes, rnn_skip_out_channels)]
        decoder_in = torch.cat((gru_out, skip_out), 1)
        
        # Output decoder layer
        decoder_out = self.decoder_net(decoder_in)       # [batch_size, out_channels]
        
        # Ar net
        if self.ar_window_size > 0:
            AR = X[:, -self.ar_window_size:, [self.target_index]]  # [batch_size, ar_window, 1]
            AR = AR.permute(0, 2, 1)                               # [batch_size, 1, ar_window]
            AR = self.ar_net(AR).squeeze(1)                        # [batch_size, out_channels]
            decoder_out = decoder_out + AR

        return decoder_out

### Test on toy dataset

In [ ]:
n_timesteps = 1000
period = 24
t = np.arange(n_timesteps)
y = (
    np.sin(2 * np.pi * t / period)
    + np.cos(0.5 * np.pi * t / period)
    + np.random.normal(scale=0.2, size=(n_timesteps,))
)

plt.figure(figsize=(10, 2.5))
plt.plot(y)
plt.grid(ls="--", lw=0.5, color="grey")

In [ ]:
# Transform into (X_train, y_train) tensors
input_seq_length = 100
out_seq_length = period * 2

n_samples = len(y) - input_seq_length - out_seq_length + 1

# Construct empty arrays to insert into
features = np.empty((n_samples, input_seq_length, 1), dtype=np.float32)
labels = np.empty((n_samples, out_seq_length, 1), dtype=np.float32)

for i in range(n_samples):
    feat_start, feat_end = i, i + input_seq_length
    features[i] = y[feat_start: feat_end].reshape(-1, 1)

    labels_start, labels_end = feat_end, feat_end + out_seq_length
    labels[i] = y[labels_start: labels_end].reshape(-1, 1)

features_ts = torch.from_numpy(features)
labels_ts = torch.from_numpy(labels)

train_ds = TensorDataset(features_ts, labels_ts)
train_dl = DataLoader(train_ds, batch_size=32)

In [ ]:
model = LSTNet(
    in_channels=1,
    out_channels=period * 2,
    cnn_out_channels=32,
    cnn_kernel_size=2,
    rnn_out_channels=64,
    rnn_skip_step_sizes=[24],
    rnn_skip_out_channels=[10],
    ar_window_size=24,
    dropout=0.3,
)

loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 100
epoch_pgbar = tqdm(range(n_epochs))
for epoch in epoch_pgbar:
    for batch_X, batch_y in train_dl:
        optimizer.zero_grad()
        
        y_hat = model(batch_X)

        # Output dimension is (batch_size, out_seq_length)
        y_hat = y_hat.unsqueeze(-1)
        loss = loss_fn(batch_y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    epoch_loss.append(loss_detach)
    epoch_pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
test_idx = np.random.choice(features_ts.size(0))

test_X = features_ts[[test_idx]]
test_y = labels_ts[[test_idx]]

model.eval()
with torch.no_grad():
    y_hat = model(test_X)

plt.plot(
    np.arange(test_X.size(1)),
    test_X.squeeze().numpy(),
    label="train",
)
plt.plot(
    np.arange(test_X.size(1), test_X.size(1) + test_y.size(1)),
    test_y.squeeze().numpy(),
    label="test",
)
plt.plot(
    np.arange(test_X.size(1), test_X.size(1) + test_y.size(1)),
    y_hat.squeeze().numpy(),
    label="predicted"
)

plt.legend()